In [1]:
import pandas as pd
import os

def batch_resample(input_dir, output_dir, src_hz=161, dst_hz=60):
    """
    批次處理資料夾內的 CSV 進行降採樣 (平滑插值法)
    """
    # 確保輸出資料夾存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已建立輸出目錄: {output_dir}")

    # 取得資料夾內所有 CSV 檔案
    files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
    
    if not files:
        print("找不到任何 CSV 檔案，請檢查路徑。")
        return

    print(f"開始處理: 來源 {src_hz}Hz -> 目標 {dst_hz}Hz")

    for file_name in files:
        input_path = os.path.join(input_dir, file_name)
        output_path = os.path.join(output_dir, f"resampled_{file_name}")

        # 1. 讀取數據
        df = pd.read_csv(input_path)
        
        # 備份原始 ID (若需要保留，但通常降採樣後 ID 會重新排序)
        # 若 CSV 有 ID 欄位則移除，避免其參與插值運算產生小數點
        if 'ID' in df.columns:
            df = df.drop(columns=['ID'])

        # 2. 建立時間索引 (基於原始頻率)
        # 每個樣本間隔 = 1 / src_hz 秒
        df.index = pd.to_timedelta(df.index / src_hz, unit='s')

        # 3. 執行重採樣 (Resample)
        # 使用 target_period = 1 / dst_hz 秒
        target_period = f"{1000/dst_hz:.6f}ms"
        
        # .mean() 聚合 + .interpolate() 線性插值，確保波形平滑且不因不整除而跳格
        resampled_df = df.resample(target_period).mean().interpolate(method='linear')

        # 4. 還原為整數索引並重新加入 ID
        resampled_df = resampled_df.reset_index(drop=True)
        resampled_df.insert(0, 'ID', range(1, len(resampled_df) + 1))

        # 5. 儲存結果
        resampled_df.to_csv(output_path, index=False)
        print(f"  - 已完成: {file_name} -> {len(resampled_df)} 筆數據")

    print("\n所有檔案批次處理完畢。")

# --- 設定參數 ---
INPUT_FOLDER = './output2'    # 您的原始數據資料夾
OUTPUT_FOLDER = './output_60hz' # 轉換後的儲存位置
SOURCE_HZ = 161               # 原始頻率
TARGET_HZ = 60                # 目標頻率

# 執行
batch_resample(INPUT_FOLDER, OUTPUT_FOLDER, SOURCE_HZ, TARGET_HZ)

已建立輸出目錄: ./output_60hz
開始處理: 來源 161Hz -> 目標 60Hz
  - 已完成: 1771838245858_nn_160hz_small_esp32_t312_v10_notTired_pre.csv -> 18796 筆數據
  - 已完成: 1771837901232_zhao_160hz_all_esp32_t83_v10_notTired_pre.csv -> 5028 筆數據
  - 已完成: 1771837908327_nn_160hz_all_esp32_t94_v10_notTired_pre.csv -> 5695 筆數據
  - 已完成: 1771838245295_zhao_160hz_small_esp32_t317_v10_notTired_pre.csv -> 19129 筆數據
  - 已完成: 1771838708821_zhao_160hz_small_esp32_t230_v10_notTired_pre.csv -> 13904 筆數據
  - 已完成: 1771839036304_zhao_160hz_stairs_esp32_t108_v10_notTired_pre.csv -> 6510 筆數據
  - 已完成: 1771838875190_nn_160hz_large_esp32_t74_v10_notTired_pre.csv -> 4462 筆數據
  - 已完成: 1771838866705_zhao_160hz_large_esp32_t69_v10_notTired_pre.csv -> 4193 筆數據
  - 已完成: 1772007820799_zhao_160hz_all_esp32_t82_v10_notTired_pre.csv -> 4948 筆數據
  - 已完成: 1771838715410_nn_160hz_small_esp32_t240_v10_notTired_pre.csv -> 14455 筆數據
  - 已完成: 1771839063755_nn_160hz_large_esp32_t173_v10_Tired_pre.csv -> 10421 筆數據
  - 已完成: 1772007833450_nn_160hz_all_esp32_t83